# Agent Design · Day 26 — **Speculative AI & Reasoning Agents**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~2.5 hrs

Yesterday you chose the *control flow* around the LLM. Today you make the LLM's **reasoning itself more reliable**. A model that reasons out loud (Chain-of-Thought) is more accurate — but it can reason confidently to a *wrong* answer. In banking, a plausible-but-wrong compliance verdict is worse than no answer. So we add a second layer: the agent **verifies its own reasoning** before committing.

Today:
1. A compliance reasoning task + a deterministic mock reasoner (keyless, reproducible).
2. **Chain-of-Thought (CoT)** — reason step by step, and see how it can still hallucinate.
3. **Chain-of-Verification (CoVe)** — draft, generate verification questions, answer them independently, revise.
4. **Extended thinking** — trading a reasoning *budget* for accuracy.
5. A **self-critique loop** that corrects a regulatory answer.

Everything runs with no API key — a `FakeReasoner` mimics how an extended-thinking model (e.g. `claude-3-7-sonnet`) exposes its reasoning trace.

## 1. A reasoning task + a mock reasoner

The task: decide whether a payment is allowed under a (toy) sanctions + limit policy. It needs several facts combined correctly. The reasoner is deterministic — same input, same trace — so the lesson is reproducible.

In [1]:
POLICY = {
    "daily_limit": 10_000,          # GBP
    "sanctioned_countries": {"XX", "YY"},
    "high_risk_over": 5_000,        # needs extra check above this
}

CASE = {
    "customer": "Priya Shah",
    "amount": 7_500,
    "dest_country": "ZZ",           # not sanctioned
    "prior_today": 4_000,           # already sent today
}

# A "fact base" the reasoner can query — the ground truth.
def fact(name):
    return {
        "daily_limit": POLICY["daily_limit"],
        "is_sanctioned": CASE["dest_country"] in POLICY["sanctioned_countries"],
        "amount": CASE["amount"],
        "prior_today": CASE["prior_today"],
        "high_risk_over": POLICY["high_risk_over"],
    }[name]

print("case:", CASE["customer"], "wants to send", CASE["amount"], "to", CASE["dest_country"])
print("already sent today:", CASE["prior_today"], "| daily limit:", POLICY["daily_limit"])

case: Priya Shah wants to send 7500 to ZZ
already sent today: 4000 | daily limit: 10000


☝ The decision hinges on **three** facts interacting: sanctions (no), the *cumulative* daily total (4000 + 7500 = 11500 > 10000 → breach), and the high-risk threshold. A single-shot answer often catches one and misses another — exactly the failure verification is built to stop.

## 2. Chain-of-Thought — reason step by step (and still miss)

**CoT** prompts the model to emit intermediate steps before the answer. It raises accuracy on multi-step problems and makes reasoning auditable. But the steps can be *individually plausible yet collectively wrong* — here the reasoner forgets to add the prior spend.

In [2]:
def cot_reason(case):
    steps = []
    steps.append(f"Step 1: destination {case['dest_country']} sanctioned? -> {fact('is_sanctioned')}")
    steps.append(f"Step 2: amount {fact('amount')} <= daily limit {fact('daily_limit')}? -> {fact('amount') <= fact('daily_limit')}")
    # BUG (a very common CoT slip): compares THIS payment to the limit,
    # forgetting the customer already spent `prior_today`.
    allowed = (not fact("is_sanctioned")) and (fact("amount") <= fact("daily_limit"))
    steps.append(f"Step 3: not sanctioned AND under limit -> ALLOW = {allowed}")
    return allowed, steps

allow, trace = cot_reason(CASE)
print("\n".join(trace))
print("CoT verdict:", "ALLOW" if allow else "BLOCK")

Step 1: destination ZZ sanctioned? -> False
Step 2: amount 7500 <= daily limit 10000? -> True
Step 3: not sanctioned AND under limit -> ALLOW = True
CoT verdict: ALLOW


☝ Every step reads as correct, and the verdict is **ALLOW** — but it's *wrong*. The reasoner checked `amount <= limit` and never added the £4,000 already sent today (11,500 > 10,000 is a breach). CoT surfaced its reasoning, which is exactly what lets the next layer catch the gap.

## 3. Chain-of-Verification — the model checks its own facts

**CoVe** runs in four moves: (1) draft an answer, (2) generate **verification questions** that would expose an error, (3) answer each question *independently* against the fact base, (4) revise the answer using the findings. The independence is the key — it stops the model from just re-confirming its first guess.

In [3]:
def cove_reason(case):
    draft, _ = cot_reason(case)                     # 1. draft (the flawed CoT answer)

    # 2. verification questions that would catch a limit-breach.
    checks = {
        "cumulative_total": fact("amount") + fact("prior_today"),
        "breaches_daily_limit": fact("amount") + fact("prior_today") > fact("daily_limit"),
        "needs_high_risk_review": fact("amount") > fact("high_risk_over"),
    }

    # 3+4. revise: block if any independent check fails.
    revised = draft and not checks["breaches_daily_limit"]
    return draft, checks, revised

draft, checks, revised = cove_reason(CASE)
print("draft verdict :", "ALLOW" if draft else "BLOCK")
for k, v in checks.items():
    print(f"  verify {k:22}: {v}")
print("revised verdict:", "ALLOW" if revised else "BLOCK", "<- correct")

draft verdict : ALLOW
  verify cumulative_total      : 11500
  verify breaches_daily_limit  : True
  verify needs_high_risk_review: True
revised verdict: BLOCK <- correct


☝ CoVe caught it. The independent check computed the **cumulative total** (11,500), saw it breaches the daily limit, and flipped the verdict to **BLOCK**. The draft was wrong; verification made the final answer right. This is the single highest-leverage reliability trick for factual agent answers — and it's cheap: a handful of targeted checks, not a bigger model.

## 4. Extended thinking — spend a reasoning budget for accuracy

Reasoning models (extended thinking) let you trade a **token budget** for depth: more budget → more verification passes → fewer errors, at higher latency and cost. You tune the budget to the stakes. Below, a tiny simulation: more `budget` unlocks more checks.

In [4]:
def reason_with_budget(case, budget):
    # budget buys verification depth (each check "costs" one unit)
    checks_available = ["sanctions", "single_limit", "cumulative_limit", "high_risk"]
    used = checks_available[:budget]
    total = fact("amount") + fact("prior_today")
    verdict = True
    if "sanctions" in used and fact("is_sanctioned"): verdict = False
    if "single_limit" in used and fact("amount") > fact("daily_limit"): verdict = False
    if "cumulative_limit" in used and total > fact("daily_limit"): verdict = False
    return verdict, used

for b in (2, 4):
    v, used = reason_with_budget(CASE, b)
    print(f"budget={b} checks={used} -> {'ALLOW' if v else 'BLOCK'}")

budget=2 checks=['sanctions', 'single_limit'] -> ALLOW
budget=4 checks=['sanctions', 'single_limit', 'cumulative_limit', 'high_risk'] -> BLOCK


☝ At `budget=2` the agent only checks sanctions and the single-payment limit → wrong **ALLOW**. At `budget=4` it reaches the cumulative-limit check → correct **BLOCK**. That's the extended-thinking trade-off in miniature: cheap tasks get a small budget, a £7,500 cross-border payment gets a large one. Match the budget to the cost of being wrong.

## 5. Self-critique loop — draft, critique, correct

Put it together as an agent loop: produce an answer, run a **critique** that hunts for the specific failure mode (incomplete reasoning), and if it fails, retry with the critique injected. This is Reflexion (Day 25) applied to *reasoning quality* rather than task completeness.

In [5]:
def self_critique_agent(case, max_iters=2):
    answer, _ = cot_reason(case)                       # first draft
    for i in range(1, max_iters + 1):
        total = fact("amount") + fact("prior_today")
        critique = None
        if answer and total > fact("daily_limit"):
            critique = f"cumulative total {total} breaches limit {fact('daily_limit')}"
        if critique is None:
            print(f"iter {i}: passed critique"); break
        print(f"iter {i}: CRITIQUE -> {critique}; correcting")
        answer = False                                 # apply the correction
    return answer

final = self_critique_agent(CASE)
print("final verdict:", "ALLOW" if final else "BLOCK")

iter 1: CRITIQUE -> cumulative total 11500 breaches limit 10000; correcting
iter 2: passed critique
final verdict: BLOCK


☝ The loop turned a wrong first draft into a defensible final answer *and left an audit trail* of why it changed its mind — the critique text is exactly what a compliance reviewer (or a regulator) wants to see. Speculative reasoning isn't about a smarter single answer; it's about **not shipping the confident wrong one**.

## 6. Recap — reliability is a layer you add, not a model you buy

| Technique | What it does | Catches | Cost |
|---|---|---|---|
| **Chain-of-Thought** | reason in explicit steps | multi-step arithmetic slips (sometimes) | +tokens |
| **Chain-of-Verification** | draft → verify facts independently → revise | plausible-but-wrong answers | a few checks |
| **Extended thinking** | budget more reasoning depth | errors from shallow reasoning | latency + tokens |
| **Self-critique loop** | critique own output, retry | incomplete / unchecked reasoning | extra passes |

The pattern under all four: **never trust a single forward pass on a high-stakes decision.** CoVe and self-critique are the cheapest reliability wins in the whole course — a handful of targeted checks turned an *ALLOW* into the correct *BLOCK*. Tomorrow (Day 27) we defend the *inputs and outputs* around this reasoner: prompt-injection and PII.